# FraudGraph — Stage 3: PureSAGE vs HybridGatedGNN ablation

We train **two models** on the same graph and compare them head-to-head:

- **PureSAGE** — every edge type uses SAGEConv. Standard fraud-GNN baseline
  (Alibaba, Ant Financial, PayPal all deploy variants of this).
- **HybridGatedGNN** — every edge type runs SAGE *and* GATv2 in parallel, mixed
  by a **learned per-node gate** with temperature. The gate is a linear layer
  over the destination node's own features, so the model learns per-transaction
  whether attention (GAT) or mean aggregation (SAGE) is more informative for
  that node's context. Trained with **focal loss** (Lin et al. 2017) — better
  than weighted-BCE on extreme imbalance because it directly downweights easy
  negatives instead of amplifying rare-positive gradients.

Both classes live in `src/model.py` — Stage 5 serving will import whichever won.

## 0a. Base installs

In [1]:
!pip install -q torch_geometric mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 87.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 84.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

## 0b. Install PyG C++ sampling backends (pyg-lib + torch-sparse)

`NeighborLoader` requires `pyg-lib` or `torch-sparse`. We auto-detect Kaggle's
torch+CUDA version and install matching prebuilt wheels. **Restart kernel
after this cell succeeds**, then Run All from the top.

In [2]:
import torch, subprocess, sys

TORCH_VER = torch.__version__.split("+")[0]
CUDA_VER  = (torch.version.cuda or "").replace(".", "")
maj, minor, patch = TORCH_VER.split(".")
print(f"detected torch={TORCH_VER}  cuda={torch.version.cuda}")
assert CUDA_VER, "no CUDA detected — enable GPU T4 in notebook Settings"

candidates = [f"{maj}.{minor}.{patch}+cu{CUDA_VER}", f"{maj}.{minor}.0+cu{CUDA_VER}"]
ok = False
for slug in candidates:
    url = f"https://data.pyg.org/whl/torch-{slug}.html"
    print(f"\ntrying {url}")
    rc = subprocess.call([sys.executable, "-m", "pip", "install", "-q",
                          "pyg-lib", "torch-sparse", "-f", url])
    if rc == 0:
        print(f"installed pyg-lib + torch-sparse from {url}")
        ok = True
        break
assert ok, f"no matching wheel found for {candidates}"

detected torch=2.10.0  cuda=12.8

trying https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 46.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 71.1 MB/s eta 0:00:00
installed pyg-lib + torch-sparse from https://data.pyg.org/whl/torch-2.10.0+cu128.html


## 0c. Regular imports + backend check

In [3]:
import os, time, pickle
import numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader
from torch_geometric.transforms import ToUndirected
from torch_geometric.nn import HeteroConv, SAGEConv, GATv2Conv
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
import mlflow

try:
    import pyg_lib; print("pyg-lib OK")
except ImportError:
    try:
        import torch_sparse; print("torch-sparse OK")
    except ImportError:
        raise RuntimeError("Neither pyg-lib nor torch-sparse loaded. Restart kernel and rerun 0b.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu")
OUT = "/kaggle/working"

pyg-lib OK
device: cuda | Tesla T4


## 1. Load graph + reverse edges

In [4]:
GRAPH_PATH = "/kaggle/input/datasets/svkayy/fraudgraph-stage2/graph.pt"
if not os.path.exists(GRAPH_PATH):
    import glob
    hits = glob.glob("/kaggle/input/**/graph.pt", recursive=True)
    assert hits, "graph.pt not found anywhere under /kaggle/input"
    GRAPH_PATH = hits[0]
print("loading:", GRAPH_PATH)

data = torch.load(GRAPH_PATH, weights_only=False)
data = ToUndirected(merge=False)(data)
print(data)

loading: /kaggle/input/datasets/svkayy/fraudgraph-stage2/graph.pt
HeteroData(
  transaction={
    x=[590540, 437],
    y=[590540],
    train_mask=[590540],
    test_mask=[590540],
  },
  card={ num_nodes=13553 },
  merchant={ num_nodes=532 },
  device={ num_nodes=1787 },
  (transaction, uses_card, card)={ edge_index=[2, 590540] },
  (transaction, at_merchant, merchant)={ edge_index=[2, 590540] },
  (transaction, on_device, device)={ edge_index=[2, 118666] },
  (transaction, velocity_cluster, transaction)={ edge_index=[2, 114439] },
  (card, rev_uses_card, transaction)={ edge_index=[2, 590540] },
  (merchant, rev_at_merchant, transaction)={ edge_index=[2, 590540] },
  (device, rev_on_device, transaction)={ edge_index=[2, 118666] },
  (transaction, rev_velocity_cluster, transaction)={ edge_index=[2, 114439] }
)


## 2. Model definitions (mirror `src/model.py`)

In [5]:
class HybridGatedConv(nn.Module):
    def __init__(self, in_dim, out_dim, heads=4, dropout=0.2):
        super().__init__()
        per_head = out_dim // heads if heads > 1 else out_dim
        self.sage = SAGEConv((in_dim, in_dim), out_dim, aggr="mean")
        self.gat  = GATv2Conv((in_dim, in_dim), per_head, heads=heads,
                              concat=(heads > 1), add_self_loops=False, dropout=dropout)
        self.gate = nn.Linear(in_dim, 1)
        self.log_temp = nn.Parameter(torch.zeros(1))

    def forward(self, x, edge_index):
        h_sage = self.sage(x, edge_index)
        h_gat  = self.gat(x, edge_index)
        x_dst = x[1] if isinstance(x, tuple) else x
        g = torch.sigmoid(self.gate(x_dst) / self.log_temp.exp())
        return g * h_gat + (1.0 - g) * h_sage

    @torch.no_grad()
    def gate_stats(self, x, edge_index):
        x_dst = x[1] if isinstance(x, tuple) else x
        g = torch.sigmoid(self.gate(x_dst) / self.log_temp.exp())
        return g.mean().item(), g.std().item()

class _Base(nn.Module):
    def __init__(self, in_dim, nc, nm, nd, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.tx_encoder = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout))
        self.card_emb     = nn.Embedding(nc, hidden_dim)
        self.merchant_emb = nn.Embedding(nm, hidden_dim)
        self.device_emb   = nn.Embedding(nd, hidden_dim)

    def _init(self, x_tx, node_idx):
        h = {"transaction": self.tx_encoder(x_tx)}
        h["card"]     = self.card_emb.weight     if node_idx.get("card")     is None else self.card_emb(node_idx["card"])
        h["merchant"] = self.merchant_emb.weight if node_idx.get("merchant") is None else self.merchant_emb(node_idx["merchant"])
        h["device"]   = self.device_emb.weight   if node_idx.get("device")   is None else self.device_emb(node_idx["device"])
        return h

class PureSAGE(_Base):
    def __init__(self, in_dim, nc, nm, nd, metadata, hidden_dim=128, out_dim=64, dropout=0.2):
        super().__init__(in_dim, nc, nm, nd, hidden_dim, dropout)
        ets = metadata[1]
        self.conv1 = HeteroConv({et: SAGEConv((hidden_dim, hidden_dim), hidden_dim, aggr="mean") for et in ets}, aggr="sum")
        self.conv2 = HeteroConv({et: SAGEConv((hidden_dim, hidden_dim), out_dim, aggr="mean") for et in ets}, aggr="sum")
        self.head  = nn.Sequential(nn.Linear(out_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))

    def encode(self, x_tx, edge_index_dict, node_idx=None):
        h = self._init(x_tx, node_idx or {})
        h = self.conv1(h, edge_index_dict); h = {k: F.relu(v) for k, v in h.items()}
        h = self.conv2(h, edge_index_dict); h = {k: F.relu(v) for k, v in h.items()}
        return h

    def forward(self, x_tx, edge_index_dict, node_idx=None):
        return self.head(self.encode(x_tx, edge_index_dict, node_idx)["transaction"]).squeeze(-1)

class HybridGatedGNN(_Base):
    def __init__(self, in_dim, nc, nm, nd, metadata, hidden_dim=128, out_dim=64, heads=4, dropout=0.2):
        super().__init__(in_dim, nc, nm, nd, hidden_dim, dropout)
        ets = metadata[1]
        self.conv1 = HeteroConv({et: HybridGatedConv(hidden_dim, hidden_dim, heads=heads, dropout=dropout) for et in ets}, aggr="sum")
        self.conv2 = HeteroConv({et: HybridGatedConv(hidden_dim, out_dim,    heads=1,     dropout=dropout) for et in ets}, aggr="sum")
        self.head  = nn.Sequential(nn.Linear(out_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))

    def encode(self, x_tx, edge_index_dict, node_idx=None):
        h = self._init(x_tx, node_idx or {})
        h = self.conv1(h, edge_index_dict); h = {k: F.relu(v) for k, v in h.items()}
        h = self.conv2(h, edge_index_dict); h = {k: F.relu(v) for k, v in h.items()}
        return h

    def forward(self, x_tx, edge_index_dict, node_idx=None):
        return self.head(self.encode(x_tx, edge_index_dict, node_idx)["transaction"]).squeeze(-1)

## 3. Loss functions

In [6]:
def weighted_bce(logits, targets, pos_weight):
    return F.binary_cross_entropy_with_logits(
        logits, targets, pos_weight=torch.tensor(pos_weight, device=logits.device))

def focal_bce(logits, targets, alpha=0.25, gamma=2.0, pos_weight=None):
    ce = F.binary_cross_entropy_with_logits(
        logits, targets, reduction="none",
        pos_weight=None if pos_weight is None else torch.tensor(pos_weight, device=logits.device))
    p = torch.sigmoid(logits)
    p_t = p * targets + (1 - p) * (1 - targets)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    return (alpha_t * (1 - p_t).pow(gamma) * ce).mean()

## 4. NeighborLoader (shared by both models)

In [7]:
train_idx = data["transaction"].train_mask.nonzero(as_tuple=False).view(-1)
test_idx  = data["transaction"].test_mask.nonzero(as_tuple=False).view(-1)
print("train tx:", train_idx.numel(), "  test tx:", test_idx.numel())

BATCH = 512
NUM_NEIGHBORS = [15, 10]

train_loader = NeighborLoader(data, num_neighbors=NUM_NEIGHBORS,
    input_nodes=("transaction", train_idx), batch_size=BATCH, shuffle=True)
_eval_gen = torch.Generator().manual_seed(42)
eval_loader = NeighborLoader(data, num_neighbors=NUM_NEIGHBORS,
    input_nodes=("transaction", test_idx), batch_size=1024, shuffle=False,
    generator=_eval_gen)
print("train batches:", len(train_loader), "  eval batches:", len(eval_loader))

pos_weight = float(((data["transaction"].y[train_idx] == 0).sum() /
                    (data["transaction"].y[train_idx] == 1).sum()))
print("pos_weight:", round(pos_weight, 1))
in_dim = data["transaction"].x.shape[1]
metadata = data.metadata()
nc, nm, nd = data["card"].num_nodes, data["merchant"].num_nodes, data["device"].num_nodes

train tx: 472432   test tx: 118108
train batches: 923   eval batches: 116
pos_weight: 27.5


## 5. Training helper

In [8]:
def evaluate(model):
    model.eval()
    probs, ys = [], []
    with torch.no_grad():
        for batch in eval_loader:
            batch = batch.to(DEVICE)
            node_idx = {t: batch[t].n_id for t in ["card", "merchant", "device"]}
            logit = model(batch["transaction"].x, batch.edge_index_dict, node_idx=node_idx)
            seed = batch["transaction"].batch_size
            probs.append(torch.sigmoid(logit[:seed]).cpu().numpy())
            ys.append(batch["transaction"].y[:seed].cpu().numpy())
    p = np.concatenate(probs); y = np.concatenate(ys)
    prec, rec, _ = precision_recall_curve(y, p)
    return {
        "auc": float(roc_auc_score(y, p)),
        "average_precision": float(average_precision_score(y, p)),
        "precision_at_10recall": float(prec[np.argmin(np.abs(rec - 0.10))]),
    }, p, y

def train_model(model, loss_fn, tag, epochs=50, eval_every=2, patience=8, lr=1e-3):
    model = model.to(DEVICE)
    # warmup so lazy convs create their params
    model.eval()
    with torch.no_grad():
        batch = next(iter(train_loader)).to(DEVICE)
        node_idx = {t: batch[t].n_id for t in ["card", "merchant", "device"]}
        _ = model(batch["transaction"].x, batch.edge_index_dict, node_idx=node_idx)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=2, min_lr=1e-5)

    history, best = [], {"average_precision": 0.0, "epoch": -1}
    best_state = None; bad = 0

    for epoch in range(1, epochs + 1):
        model.train()
        t0 = time.time()
        total_loss, seen = 0.0, 0
        for batch in train_loader:
            batch = batch.to(DEVICE)
            node_idx = {t: batch[t].n_id for t in ["card", "merchant", "device"]}
            opt.zero_grad()
            logit = model(batch["transaction"].x, batch.edge_index_dict, node_idx=node_idx)
            seed = batch["transaction"].batch_size
            loss = loss_fn(logit[:seed], batch["transaction"].y[:seed].float())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total_loss += loss.item() * seed; seen += seed
        avg_loss = total_loss / seen

        line = f"[{tag}] epoch {epoch:02d}  loss {avg_loss:.4f}  time {time.time()-t0:.1f}s"
        if epoch % eval_every == 0 or epoch == epochs:
            m, _, _ = evaluate(model)
            sched.step(m["average_precision"])
            line += f"   AUC {m['auc']:.4f}  AP {m['average_precision']:.4f}  P@10R {m['precision_at_10recall']:.4f}"
            history.append({"epoch": epoch, "loss": avg_loss, **m})
            if m["average_precision"] > best["average_precision"] + 1e-4:
                best = {"epoch": epoch, **m}
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                bad = 0
            else:
                bad += 1
                if bad >= patience:
                    print(line); print(f"[{tag}] early stop @ epoch {epoch}")
                    break
        print(line)

    model.load_state_dict(best_state)
    final, p_test, y_test = evaluate(model)
    print(f"\n[{tag}] final: {final}   best epoch: {best['epoch']}")
    return model, best_state, final, history, p_test, y_test

## 6. Train Model A — PureSAGE (weighted BCE)

Baseline. All 8 edge types use plain SAGEConv. Weighted BCE with pos_weight=28.
This is what the fraud-detection playbook actually deploys — reproducing it as
a control before claiming the hybrid helps.

In [9]:
torch.manual_seed(0)
model_A = PureSAGE(in_dim, nc, nm, nd, metadata, hidden_dim=128, out_dim=64, dropout=0.2)
model_A, state_A, final_A, hist_A, pA, yA = train_model(
    model_A,
    loss_fn=lambda l, t: weighted_bce(l, t, pos_weight),
    tag="SAGE", epochs=50, eval_every=2, patience=8,
)

[SAGE] epoch 01  loss 0.8300  time 42.5s
[SAGE] epoch 02  loss 0.6738  time 42.6s   AUC 0.8570  AP 0.2792  P@10R 0.3571
[SAGE] epoch 03  loss 0.5916  time 42.8s
[SAGE] epoch 04  loss 0.5383  time 42.1s   AUC 0.8560  AP 0.3707  P@10R 0.5953
[SAGE] epoch 05  loss 0.4954  time 41.8s
[SAGE] epoch 06  loss 0.4644  time 41.5s   AUC 0.8543  AP 0.3996  P@10R 0.7061
[SAGE] epoch 07  loss 0.4358  time 41.7s
[SAGE] epoch 08  loss 0.4171  time 41.4s   AUC 0.8550  AP 0.3965  P@10R 0.8306
[SAGE] epoch 09  loss 0.3973  time 41.6s
[SAGE] epoch 10  loss 0.3826  time 41.6s   AUC 0.8472  AP 0.4421  P@10R 0.8962
[SAGE] epoch 11  loss 0.3639  time 41.7s
[SAGE] epoch 12  loss 0.3591  time 41.5s   AUC 0.8615  AP 0.4918  P@10R 0.9420
[SAGE] epoch 13  loss 0.3376  time 42.0s
[SAGE] epoch 14  loss 0.3365  time 42.0s   AUC 0.8464  AP 0.2552  P@10R 0.3179
[SAGE] epoch 15  loss 0.3228  time 41.8s
[SAGE] epoch 16  loss 0.3138  time 41.9s   AUC 0.8476  AP 0.3086  P@10R 0.4139
[SAGE] epoch 17  loss 0.3071  time 41.7s

## 7. Train Model B — HybridGatedGNN (focal loss)

Every edge type runs SAGE + GAT in parallel, gated per destination-node.
Focal loss handles imbalance without the aggressive-gradient spikes that
made weighted-BCE unstable. Watch the printed gate mean/std at the end — if
mean drifts away from 0.5 the model is committing to one aggregation mode.

In [10]:
torch.manual_seed(0)
model_B = HybridGatedGNN(in_dim, nc, nm, nd, metadata, hidden_dim=128, out_dim=64, heads=4, dropout=0.2)
model_B, state_B, final_B, hist_B, pB, yB = train_model(
    model_B,
    loss_fn=lambda l, t: focal_bce(l, t, alpha=0.25, gamma=2.0, pos_weight=pos_weight),
    tag="HYBRID", epochs=50, eval_every=2, patience=8,
)

[HYBRID] epoch 01  loss 0.0909  time 73.1s
[HYBRID] epoch 02  loss 0.0742  time 73.3s   AUC 0.8649  AP 0.2533  P@10R 0.2806
[HYBRID] epoch 03  loss 0.0663  time 73.3s
[HYBRID] epoch 04  loss 0.0609  time 73.2s   AUC 0.8739  AP 0.4877  P@10R 0.9598
[HYBRID] epoch 05  loss 0.0570  time 74.0s
[HYBRID] epoch 06  loss 0.0533  time 73.3s   AUC 0.8757  AP 0.3888  P@10R 0.6152
[HYBRID] epoch 07  loss 0.0505  time 73.7s
[HYBRID] epoch 08  loss 0.0479  time 73.3s   AUC 0.8646  AP 0.2652  P@10R 0.2886
[HYBRID] epoch 09  loss 0.0462  time 72.9s
[HYBRID] epoch 10  loss 0.0443  time 73.4s   AUC 0.8558  AP 0.2668  P@10R 0.2851
[HYBRID] epoch 11  loss 0.0375  time 74.1s
[HYBRID] epoch 12  loss 0.0350  time 73.3s   AUC 0.8586  AP 0.3472  P@10R 0.4845
[HYBRID] epoch 13  loss 0.0336  time 73.5s
[HYBRID] epoch 14  loss 0.0327  time 73.2s   AUC 0.8647  AP 0.3177  P@10R 0.4251
[HYBRID] epoch 15  loss 0.0315  time 73.2s
[HYBRID] epoch 16  loss 0.0303  time 73.4s   AUC 0.8601  AP 0.2761  P@10R 0.2865
[HYBRID]

### Gate statistics (hybrid model only)

In [11]:
# Peek at what the gates converged to on a real batch — gate=1 means the
# model chose GAT-style attention for that node; gate=0 means SAGE mean.
model_B.eval()
with torch.no_grad():
    batch = next(iter(eval_loader)).to(DEVICE)
    node_idx = {t: batch[t].n_id for t in ["card", "merchant", "device"]}
    h = model_B._init(batch["transaction"].x, node_idx)
    for et, conv in model_B.conv1.convs.items():
        src, _, dst = et
        x_src = h[src]; x_dst = h[dst]
        x_in = (x_src, x_dst) if src != dst else x_src
        m, s = conv.gate_stats(x_in, batch.edge_index_dict[et])
        temp = conv.log_temp.exp().item()
        print(f"  layer1  {et}  gate mean={m:.2f}  std={s:.2f}  temp={temp:.2f}")

  layer1  ('transaction', 'uses_card', 'card')  gate mean=0.33  std=0.08  temp=0.61
  layer1  ('transaction', 'at_merchant', 'merchant')  gate mean=0.45  std=0.08  temp=0.92
  layer1  ('transaction', 'on_device', 'device')  gate mean=0.44  std=0.05  temp=0.95
  layer1  ('transaction', 'velocity_cluster', 'transaction')  gate mean=0.72  std=0.11  temp=1.25
  layer1  ('card', 'rev_uses_card', 'transaction')  gate mean=0.55  std=0.31  temp=0.86
  layer1  ('merchant', 'rev_at_merchant', 'transaction')  gate mean=0.52  std=0.22  temp=1.04
  layer1  ('device', 'rev_on_device', 'transaction')  gate mean=0.84  std=0.11  temp=1.00
  layer1  ('transaction', 'rev_velocity_cluster', 'transaction')  gate mean=0.75  std=0.13  temp=1.15


## 8. Side-by-side comparison

In [12]:
print(f"{'model':16s} {'AUC':>7s} {'AP':>7s} {'P@10R':>7s}")
print("-" * 40)
for name, m in [("xgb_lean", {"auc": 0.7689, "average_precision": 0.1391, "precision_at_10recall": 0.2495}),
                ("xgb_full", {"auc": 0.9067, "average_precision": 0.5241, "precision_at_10recall": 0.9464}),
                ("PureSAGE",  final_A),
                ("Hybrid",    final_B)]:
    print(f"{name:16s} {m['auc']:7.4f} {m['average_precision']:7.4f} {m['precision_at_10recall']:7.4f}")

winner_tag = "HYBRID" if final_B["average_precision"] > final_A["average_precision"] else "SAGE"
print(f"\nBest GNN (by AP): {winner_tag}")

model                AUC      AP   P@10R
----------------------------------------
xgb_lean          0.7689  0.1391  0.2495
xgb_full          0.9067  0.5241  0.9464
PureSAGE          0.8622  0.4934  0.9486
Hybrid            0.8736  0.4876  0.9575

Best GNN (by AP): SAGE


## 9. Log both runs to MLflow

In [13]:
mlflow.set_tracking_uri(f"sqlite:///{OUT}/mlflow.db")
mlflow.set_experiment("fraudgraph")

for name, final, model, hist, loss_name in [
    ("graphsage_pure",  final_A, model_A, hist_A, "weighted_bce"),
    ("graphsage_hybrid_gated", final_B, model_B, hist_B, "focal_bce(a=0.25,g=2.0)"),
]:
    with mlflow.start_run(run_name=name):
        mlflow.log_params({
            "arch": type(model).__name__,
            "loss": loss_name,
            "hidden_dim": 128, "out_dim": 64, "dropout": 0.2,
            "lr": 1e-3, "weight_decay": 1e-5,
            "batch_size": BATCH, "num_neighbors": str(NUM_NEIGHBORS),
            "pos_weight": round(pos_weight, 2),
            "best_epoch": hist[-1]["epoch"] if hist else -1,
        })
        mlflow.log_metrics(final)
        for h in hist:
            mlflow.log_metric("val_ap", h["average_precision"], step=h["epoch"])
            mlflow.log_metric("val_auc", h["auc"], step=h["epoch"])
    print("logged:", name)

2026/07/02 05:55:43 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/02 05:55:43 INFO mlflow.store.db.utils: Updating database tables
2026/07/02 05:55:45 INFO mlflow.tracking.fluent: Experiment with name 'fraudgraph' does not exist. Creating a new experiment.


logged: graphsage_pure
logged: graphsage_hybrid_gated


## 10. Precompute node embeddings from the winner

Full-graph forward pass, dump `card` / `merchant` / `device` embeddings after
layer 2. Stage 5 uses these as cached neighbor embeddings so the fresh
transaction's inference stays under 100ms.

In [14]:
winner_model = model_B if winner_tag == "HYBRID" else model_A
winner_state = state_B  if winner_tag == "HYBRID" else state_A
winner_model.eval()
with torch.no_grad():
    x_tx = data["transaction"].x.to(DEVICE)
    edge_index_dict = {et: data[et].edge_index.to(DEVICE) for et in data.edge_types}
    h = winner_model.encode(x_tx, edge_index_dict, node_idx=None)
    for k, v in h.items():
        print(f"  {k}: {tuple(v.shape)}")
    embeddings = {k: v.detach().cpu() for k, v in h.items() if k != "transaction"}

  card: (13553, 64)
  merchant: (532, 64)
  device: (1787, 64)
  transaction: (590540, 64)


## 11. Save artifacts

In [15]:
torch.save(state_A, f"{OUT}/graphsage_pure.pt")
torch.save(state_B, f"{OUT}/graphsage_hybrid.pt")
torch.save(embeddings, f"{OUT}/node_embeddings.pt")

model_meta = {
    "winner": winner_tag,
    "final_pure":    final_A,
    "final_hybrid":  final_B,
    "in_dim": in_dim,
    "num_cards": nc, "num_merchants": nm, "num_devices": nd,
    "metadata": metadata,
    "hidden_dim": 128, "out_dim": 64, "heads": 4, "dropout": 0.2,
    "history_pure":   hist_A,
    "history_hybrid": hist_B,
}
with open(f"{OUT}/model_meta.pkl", "wb") as f:
    pickle.dump(model_meta, f)

print("Saved:")
print("  graphsage_pure.pt       (PureSAGE weights)")
print("  graphsage_hybrid.pt     (HybridGatedGNN weights)")
print("  node_embeddings.pt      (from winner, for Stage 5 serving)")
print("  model_meta.pkl          (both runs + full history)")
print("  mlflow.db               (both runs + XGBoost runs)")
print(f"\nWinner (by AP): {winner_tag}")

Saved:
  graphsage_pure.pt       (PureSAGE weights)
  graphsage_hybrid.pt     (HybridGatedGNN weights)
  node_embeddings.pt      (from winner, for Stage 5 serving)
  model_meta.pkl          (both runs + full history)
  mlflow.db               (both runs + XGBoost runs)

Winner (by AP): SAGE
